# Kural Tabanlı Sınıflandırma ile Potansiyel Müşteri Getirisi Hesaplama

## İş Problemi

Bir oyun şirketi müşterilerinin bazı özelliklerini kullanarak seviye tabanlı (level based) yeni müşteri tanımları (persona) oluşturmak ve bu yeni müşteri tanımlarına göre segmentler oluşturup bu segmentlere göre yeni gelebilecek müşterilerin şirkete ortalama ne kadar kazandırabileceğini tahmin etmek istemektedir.

Örneğin: Türkiye'den IOS kullanıcısı olan 25 yaşındaki bir erkek kullanıcının ortalama ne kadar kazandırabileceği belirlenmek isteniyor.

## Veri Seti Hikayesi

Persona.csv veri seti uluslararası bir oyun şirketinin sattığı ürünlerin fiyatlarını ve bu ürünleri satın alan kullanıcıların bazı demografik bilgilerini barındırmaktadır. Veri seti her satış işleminde oluşan kayıtlardan meydana gelmektedir. Bunun anlamı tablo tekilleştirilmemiştir. Diğer bir ifade ile belirli demografik özelliklere sahip bir kullanıcı birden fazla alışveriş yapmış olabilir.

- **Price:** Müşterinin harcama tutarı
- **Source:** Müşterinin bağlandığı cihaz türü
- **Sex:** Müşterinin cinsiyeti
- **Country:** Müşterinin ülkesi
- **Age:** Müşterinin yaşı

### Uygulama Öncesi

```
   PRICE   SOURCE   SEX COUNTRY  AGE
0     39  android  male     bra   17
1     39  android  male     bra   17
2     49  android  male     bra   17
3     29  android  male     tur   17
4     49  android  male     tur   17
```

### Uygulama Sonrası

```
       customers_level_based        PRICE SEGMENT
0   BRA_ANDROID_FEMALE_0_18  1139.800000       A
1  BRA_ANDROID_FEMALE_19_23  1070.600000       A
2  BRA_ANDROID_FEMALE_24_30   508.142857       A
3  BRA_ANDROID_FEMALE_31_40   233.166667       C
4  BRA_ANDROID_FEMALE_41_66   236.666667       C
```

## GÖREV 1: Aşağıdaki soruları yanıtlayınız.

**Soru 1:** persona.csv dosyasını okutunuz ve veri seti ile ilgili genel bilgileri gösteriniz.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
df = pd.read_csv("datasets/persona.csv")
def check_df(dataframe, head=5):
    print("##################### Shape #####################")
    print(dataframe.shape) # satır ve sütun sayısı
    print("##################### Types #####################")
    print(dataframe.dtypes) # veri tipleri
    print("##################### Head #####################")
    print(dataframe.head(head)) # ilk 5 satır
    print("##################### Tail #####################")
    print(dataframe.tail(head)) # son 5 satır
    print("##################### NA #####################")
    print(dataframe.isnull().sum()) # NA değerlerinin sayısı
    print("##################### Quantiles #####################")
    print(dataframe.describe([0, 0.05, 0.50, 0.95, 0.99, 1]).T) # istatistiksel özet

check_df(df)

##################### Shape #####################
(5000, 5)
##################### Types #####################
PRICE      int64
SOURCE       str
SEX          str
COUNTRY      str
AGE        int64
dtype: object
##################### Head #####################
   PRICE   SOURCE   SEX COUNTRY  AGE
0     39  android  male     bra   17
1     39  android  male     bra   17
2     49  android  male     bra   17
3     29  android  male     tur   17
4     49  android  male     tur   17
##################### Tail #####################
      PRICE   SOURCE     SEX COUNTRY  AGE
4995     29  android  female     bra   31
4996     29  android  female     bra   31
4997     29  android  female     bra   31
4998     39  android  female     bra   31
4999     29  android  female     bra   31
##################### NA #####################
PRICE      0
SOURCE     0
SEX        0
COUNTRY    0
AGE        0
dtype: int64
##################### Quantiles #####################
        count     mean        std   min 

**Soru 2:** Kaç unique SOURCE vardır? Frekansları nedir?

In [3]:
print("Unique SOURCE count:", df["SOURCE"].nunique())
print("SOURCE frequencies:\n", df["SOURCE"].value_counts())

Unique SOURCE count: 2
SOURCE frequencies:
 SOURCE
android    2974
ios        2026
Name: count, dtype: int64


**Soru 3:** Kaç unique PRICE vardır?

In [4]:
df["PRICE"].nunique()

6

**Soru 4:** Hangi PRICE'dan kaçar tane satış gerçekleşmiş?

In [5]:
df["PRICE"].value_counts()

PRICE
29    1305
39    1260
49    1031
19     992
59     212
9      200
Name: count, dtype: int64

**Soru 5:** Hangi ülkeden kaçar tane satış olmuş?

In [6]:
df["COUNTRY"].value_counts()

COUNTRY
usa    2065
bra    1496
deu     455
tur     451
fra     303
can     230
Name: count, dtype: int64

**Soru 6:** Ülkelere göre satışlardan toplam ne kadar kazanılmış?

In [7]:
df.groupby("COUNTRY")["PRICE"].sum()

COUNTRY
bra    51354
can     7730
deu    15485
fra    10177
tur    15689
usa    70225
Name: PRICE, dtype: int64

**Soru 7:** SOURCE türlerine göre satış sayıları nedir?

In [8]:
df.groupby("SOURCE")["PRICE"].count()

SOURCE
android    2974
ios        2026
Name: PRICE, dtype: int64

**Soru 8:** Ülkelere göre PRICE ortalamaları nedir?

In [9]:
df.groupby("COUNTRY")["PRICE"].mean()

COUNTRY
bra    34.327540
can    33.608696
deu    34.032967
fra    33.587459
tur    34.787140
usa    34.007264
Name: PRICE, dtype: float64

**Soru 9:** SOURCE'lara göre PRICE ortalamaları nedir?

In [10]:
df.groupby("SOURCE")["PRICE"].mean()

SOURCE
android    34.174849
ios        34.069102
Name: PRICE, dtype: float64

**Soru 10:** COUNTRY-SOURCE kırılımında PRICE ortalamaları nedir?

In [11]:
df.groupby(["COUNTRY", "SOURCE"])["PRICE"].mean()

COUNTRY  SOURCE 
bra      android    34.387029
         ios        34.222222
can      android    33.330709
         ios        33.951456
deu      android    33.869888
         ios        34.268817
fra      android    34.312500
         ios        32.776224
tur      android    36.229437
         ios        33.272727
usa      android    33.760357
         ios        34.371703
Name: PRICE, dtype: float64

GÖREV 2: COUNTRY, SOURCE, SEX, AGE kırılımında ortalama kazançlar nedir?

In [12]:
df.groupby(["COUNTRY", "SOURCE", "SEX", "AGE"]).agg({"PRICE": "mean"})

PRICE
COUNTRY SOURCE  SEX    AGE           
bra     android female 15   38.714286
                       16   35.944444
                       17   35.666667
                       18   32.255814
                       19   35.206897
...                               ...
usa     ios     male   42   30.250000
                       50   39.000000
                       53   34.000000
                       55   29.000000
                       59   46.500000

[348 rows x 1 columns]

GÖREV 3: Çıktıyı PRICE'a göre sıralayınız.

Önceki sorudaki çıktıyı daha iyi görebilmek için `sort_values` metodunu azalan olacak şekilde PRICE'a uygulayınız. Çıktıyı `agg_df` olarak kaydediniz.

In [14]:
agg_df = df.groupby(["COUNTRY", "SOURCE", "SEX", "AGE"]).agg({"PRICE": "mean"}).sort_values("PRICE", ascending=False)
agg_df

PRICE
COUNTRY SOURCE  SEX    AGE       
bra     android male   46    59.0
usa     android male   36    59.0
fra     android female 24    59.0
usa     ios     male   32    54.0
deu     android female 36    49.0
...                           ...
usa     ios     female 38    19.0
                       30    19.0
can     android female 27    19.0
fra     android male   18    19.0
deu     android male   26     9.0

[348 rows x 1 columns]

GÖREV 4: Indekste yer alan isimleri değişken ismine çeviriniz.

Üçüncü sor unun çıktısında yer alan PRICE dışındaki tüm değişkenler index isimleridir. Bu isimleri değişken isimlerine çeviriniz.

İpucu: `reset_index()`


In [16]:
agg_df = agg_df.reset_index()
agg_df

,index,COUNTRY,SOURCE,SEX,AGE,PRICE
0,0,bra,android,male,46,59.0
1,1,usa,android,male,36,59.0
2,2,fra,android,female,24,59.0
3,3,usa,ios,male,32,54.0
4,4,deu,android,female,36,49.0
...,...,...,...,...,...,...
343,343,usa,ios,female,38,19.0
344,344,usa,ios,female,30,19.0
345,345,can,android,female,27,19.0
346,346,fra,android,male,18,19.0


GÖREV 5: AGE değişkenini kategorik değişkene çeviriniz ve agg_df'e ekleyiniz.

Age sayısal değişkenini kategorik değişkene çeviriniz. Aralıkları ikna edici olacağını düşündüğünüz şekilde oluşturunuz.

Örneğin: `'0_18'`, `'19_23'`, `'24_30'`, `'31_40'`, `'41_70'`

In [17]:
bins = [0, 18, 23, 30, 40, 70]
labels = ['0_18', '19_23', '24_30', '31_40', '41_70']
agg_df['AGE_CAT'] = pd.cut(agg_df['AGE'], bins=bins, labels=labels, right=True)
agg_df.head()

,index,COUNTRY,SOURCE,SEX,AGE,PRICE,AGE_CAT
0,0,bra,android,male,46,59.0,41_70
1,1,usa,android,male,36,59.0,31_40
2,2,fra,android,female,24,59.0,24_30
3,3,usa,ios,male,32,54.0,31_40
4,4,deu,android,female,36,49.0,31_40


GÖREV 6: Yeni level based müşterileri tanımlayınız ve veri setine değişken olarak ekleyiniz.

`customers_level_based` adında bir değişken tanımlayınız ve veri setine bu değişkeni ekleyiniz.

**Dikkat!**
- List comp ile `customers_level_based` değerleri oluşturulduktan sonra bu değerlerin tekilleştirilmesi gerekmektedir.
- Örneğin birden fazla şu ifadeden olabilir: `USA_ANDROID_MALE_0_18`
- Bunları groupby'a alıp price ortalamalarını almak gerekmektedir.

In [18]:
agg_df['customers_level_based'] = (
    agg_df['COUNTRY'].str.upper() + "_" +
    agg_df['SOURCE'].str.upper() + "_" +
    agg_df['SEX'].str.upper() + "_" +
    agg_df['AGE_CAT'].astype(str)
)

agg_df = agg_df.groupby('customers_level_based').agg({'PRICE': 'mean'}).reset_index()

agg_df.head()

,customers_level_based,PRICE
0,BRA_ANDROID_FEMALE_0_18,35.645303
1,BRA_ANDROID_FEMALE_19_23,34.077340
2,BRA_ANDROID_FEMALE_24_30,33.863946
3,BRA_ANDROID_FEMALE_31_40,34.898326
4,BRA_ANDROID_FEMALE_41_70,36.737179


GÖREV 7: Yeni müşterileri (USA_ANDROID_MALE_0_18) segmentlere ayırınız.

- PRICE'a göre segmentlere ayırınız
- Segmentleri "SEGMENT" isimlendirmesi ile agg_df'e ekleyiniz
- Segmentleri betimleyiniz

In [19]:
agg_df['SEGMENT'] = pd.qcut(agg_df['PRICE'], 4, labels=['D', 'C', 'B', 'A'])

print(agg_df.groupby('SEGMENT').agg({'PRICE': ['mean', 'max', 'min', 'sum', 'count']}))

agg_df.head()

             PRICE                                         
              mean        max        min          sum count
SEGMENT                                                    
D        29.206780  32.333333  19.000000   817.789833    28
C        33.509674  34.077340  32.500000   904.761209    27
B        34.999645  36.000000  34.103727   944.990411    27
A        38.691234  45.428571  36.060606  1044.663328    27


,customers_level_based,PRICE,SEGMENT
0,BRA_ANDROID_FEMALE_0_18,35.645303,B
1,BRA_ANDROID_FEMALE_19_23,34.077340,C
2,BRA_ANDROID_FEMALE_24_30,33.863946,C
3,BRA_ANDROID_FEMALE_31_40,34.898326,B
4,BRA_ANDROID_FEMALE_41_70,36.737179,A


GÖREV 8: Yeni gelen müşterileri sınıflandırınız, ne kadar gelir getirebileceğini tahmin ediniz.

**Soru 1:** 33 yaşında ANDROID kullanan bir Türk kadını hangi segmente aittir ve ortalama ne kadar gelir kazandırması beklenir?

**Soru 2:** 35 yaşında IOS kullanan bir Fransız kadını hangi segmente aittir ve ortalama ne kadar gelir kazandırması beklenir?

In [20]:

# Soru 1: 33 yaşında ANDROID kullanan bir Türk kadını
new_user1 = 'TUR_ANDROID_FEMALE_31_40'
result1 = agg_df[agg_df['customers_level_based'] == new_user1]

if not result1.empty:
    segment1 = result1['SEGMENT'].values[0]
    price1 = result1['PRICE'].values[0]
    print(f"33 yaşında ANDROID kullanan bir Türk kadını segmenti: {segment1}, beklenen gelir: {price1:.2f}")
else:
    print("Kullanıcı tanımı bulunamadı.")


33 yaşında ANDROID kullanan bir Türk kadını segmenti: A, beklenen gelir: 41.83


In [21]:
# Soru 2: 35 yaşında IOS kullanan bir Fransız kadını
new_user2 = 'FRA_IOS_FEMALE_31_40'
result2 = agg_df[agg_df['customers_level_based'] == new_user2]

if not result2.empty:
    segment2 = result2['SEGMENT'].values[0]
    price2 = result2['PRICE'].values[0]
    print(f"35 yaşında IOS kullanan bir Fransız kadını segmenti: {segment2}, beklenen gelir: {price2:.2f}")
else:
    print("Kullanıcı tanımı bulunamadı.")


35 yaşında IOS kullanan bir Fransız kadını segmenti: C, beklenen gelir: 32.82
